## Project 1 — MockMaster

An interactive chatbot that tests student understanding across **5 lesson documents**. Each lesson generates **5 questions** from the uploaded material and evaluates the student's answers with grading and tutor feedback.

### Architecture

| Module | Purpose | 
|--------|---------|
| [`read_document.py`](read_document.py) | Vector store config for 5 lesson documents |
| [`generate_questions.py`](generate_questions.py) | Quiz generation agent — 5 Qs per lesson |(`hostedtools.ipynb`) |
| [`quiz_assessment.py`](quiz_assessment.py) | Answer grading function tool | `@function_tool` (`functionastool.ipynb`) |
| [`assessment_result.py`](assessment_result.py) | Tutor feedback agent exposed as tool | `Agent.as_tool()` (`agentsastools1.ipynb`) |
| [`quiz_chatbot.py`](quiz_chatbot.py) | Orchestrator — interactive quiz chatbot | Combines all three patterns |

### Setup
1. Upload each lesson PDF to **OpenAI Dashboard → Files → Vector Stores**.
2. Set vector store IDs in `.env` (`LESSON_1_VECTOR_STORE_ID` … `LESSON_5_VECTOR_STORE_ID`) or edit `read_document.py`.
3. Run: `python quiz_chatbot.py`

This notebook demonstrates each module individually below.

### Step 1 — Imports & Document Configuration (`read_document.py`)

In [ ]:
from read_document import (
    client, MODEL, TOTAL_LESSONS, QUESTIONS_PER_LESSON,
    LESSON_VECTOR_STORES, get_vector_store_id, list_configured_lessons,
)

print(f"Model: {MODEL}")
print(f"Total lessons: {TOTAL_LESSONS}, Questions per lesson: {QUESTIONS_PER_LESSON}")
print(f"Configured lessons: {list_configured_lessons()}")
print(f"\nVector store mapping:")
for num, sid in LESSON_VECTOR_STORES.items():
    status = "configured" if "REPLACE" not in sid else "NOT configured"
    print(f"  Lesson {num}: {sid[:30]}... ({status})")

### Step 2 — Generate Quiz Questions for a Lesson (`generate_questions.py`)

Uses `FileSearchTool` to retrieve content from the lesson's vector store and generates 5 questions with reference answers.

In [ ]:
import json
from generate_questions import generate_lesson_questions, get_lesson_session, SESSION

# Generate questions for Lesson 1 (change lesson number as needed)
LESSON_TO_TEST = 1

quiz_data = await generate_lesson_questions(LESSON_TO_TEST)
print(json.dumps(quiz_data, indent=2))

### Step 3 — Grade an Answer (`quiz_assessment.py`)

Uses `@function_tool` pattern: compares the student's answer to the stored reference via an LLM grading call.

In [ ]:
from quiz_assessment import assess_answer

# Display question 1 from the generated lesson
session = get_lesson_session(LESSON_TO_TEST)
print(f"Q1: {session['questions'][0]}\n")

# Simulate a student answer (change this to test different responses)
sample_student_answer = "I think it was the registrar's office"

# The assess_answer tool is used by the orchestrator agent, but we can test it directly:
from agents import Runner, trace

from quiz_chatbot import quiz_orchestrator

GRADE_SINGLE_ANSWER_PROMPT = (
    f"Lesson {LESSON_TO_TEST}, Question 1. "
    f"The student's answer is: \"{sample_student_answer}\". "
    f"Please grade it using the assess_answer tool."
)

with trace(f"lesson_{LESSON_TO_TEST}_sample_grading"):
    result = await Runner.run(quiz_orchestrator, GRADE_SINGLE_ANSWER_PROMPT)
print(result.final_output)

### Step 4 — Get Detailed Tutor Feedback (`assessment_result.py`)

Uses `Agent.as_tool()` pattern: the tutor agent provides deeper educational feedback when the student wants more explanation.

In [ ]:
from assessment_result import tutor_feedback_tool

REQUEST_TUTOR_FEEDBACK_PROMPT = (
    f"The student wants a deeper explanation for Lesson {LESSON_TO_TEST}, Question 1.\n"
    f"Question: {session['questions'][0]}\n"
    f"Reference answer: {session['reference_answers'][0]}\n"
    f"Student's answer: {sample_student_answer}\n"
    f"Please use the provide_detailed_feedback tool."
)

with trace(f"lesson_{LESSON_TO_TEST}_tutor_feedback"):
    feedback_result = await Runner.run(quiz_orchestrator, REQUEST_TUTOR_FEEDBACK_PROMPT)
print(feedback_result.final_output)

### Step 5 — Run the Full Interactive Quiz (`quiz_chatbot.py`)

The chatbot walks students through all configured lessons interactively. Run from terminal:

```bash
python quiz_chatbot.py
```

Or test a single lesson programmatically below:

### Reference — Agentic Patterns Used

| Module | Pattern | Course Demo |
|--------|---------|-------------|
| `read_document.py` / `generate_questions.py` | `FileSearchTool` + vector store | `4_AgenticPatterns/tools/hostedtools.ipynb` |
| `quiz_assessment.py` | `@function_tool` decorator | `4_AgenticPatterns/tools/functionastool.ipynb` |
| `assessment_result.py` | `Agent.as_tool()` | `4_AgenticPatterns/tools/agentsastools1.ipynb` |
| `quiz_chatbot.py` | Orchestrator combining all tools | All three patterns above |

In [ ]:
from quiz_chatbot import run_lesson_quiz

# Run the interactive quiz for a single lesson (will prompt for input)
# await run_lesson_quiz(LESSON_TO_TEST)